#### Problem 1: Generating Random Boolean Functions

##### The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) is designed to work with functions that accept a fixed number of [Boolean inputs](https://realpython.com/python-boolean/) and return a single [Boolean output](https://realpython.com/python-boolean/). Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

<div style="font-size: 0.92em;">

### How I will complete this problem

#### Goal
Build `random_constant_balanced()` so it returns a callable `f(a, b, c, d) -> bool` that is guaranteed to be constant or balanced.

#### Function Documentation (API)
- **Name:** `random_constant_balanced`
- **Parameters:** none
- **Returns:** callable `(a: bool, b: bool, c: bool, d: bool) -> bool`
- **Guarantee:** output function always satisfies the Deutsch–Jozsa promise
- **Libraries used:** Python standard library only (`itertools`, `math`, `random`)

#### Step-by-step plan
1. Generate all 16 possible 4-input Boolean tuples.
2. Count valid function families: 2 constant and `C(16, 8)` balanced.
3. Use a random draw to select which family to construct.
4. Return always-`False` or always-`True` for constant cases.
5. For balanced case, choose exactly 8 tuples that map to `True`.
6. Store chosen tuples in a `set` for fast membership checks.
7. Return an inner function that checks membership in the tuple set.

#### Why this is correct
- Constant path returns the same value for every input.
- Balanced path marks exactly 8 of 16 inputs as `True`.
- Therefore every returned function is valid under the problem promise.

#### Complexity notes
- Construction: constant-size work for fixed 4-bit input space.
- Evaluation of returned function: average `O(1)` set lookup.

#### Example use
```python
f = random_constant_balanced()
print(f(False, False, False, False))
print(f(True, False, True, False))
```

#### Testing checklist
- Constant mode can produce both always-`False` and always-`True`.
- Balanced mode yields exactly 8 `True` values over all 16 inputs.
- Returned function accepts exactly 4 Boolean arguments.

</div>

In [ ]:
import itertools
import math
import random


def random_constant_balanced():
    # Step 1: enumerate all 16 possible inputs for 4 Boolean variables.
    all_inputs = list(itertools.product([False, True], repeat=4))

    # Step 2: choose uniformly from all valid functions:
    # - 2 constant functions
    # - C(16, 8) balanced functions
    num_balanced = math.comb(16, 8)
    total_valid = 2 + num_balanced
    draw = random.randrange(total_valid)

    # Step 3: return a constant function when selected.
    if draw == 0:
        return lambda a, b, c, d: False
    if draw == 1:
        return lambda a, b, c, d: True

    # Step 4: build a balanced function by choosing exactly 8
    # input combinations that should map to True.
    true_inputs = set(random.sample(all_inputs, 8))

    # Step 5: return a callable (a, b, c, d) using that mapping.
    def f(a, b, c, d):
        return (a, b, c, d) in true_inputs

    return f

# Output values from the original function example with explanation.
f = random_constant_balanced()
input_a = (False, False, False, False)
input_b = (True, False, True, False)
out_a = f(*input_a)
out_b = f(*input_b)

print(f"f{input_a} = {out_a}")
print(f"f{input_b} = {out_b}")

if out_a == out_b:
    print("These two sampled outputs are the same.")
else:
    print("These two sampled outputs are different.")
print("Note: two samples alone do not prove constant vs balanced.")

f(False, False, False, False) = True
f(True, False, True, False) = False
These two sampled outputs are different.
Note: two samples alone do not prove constant vs balanced.


#### Problem 2: Classical Testing for Function Type

##### [Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) is designed to demonstrate a [potential advantage of quantum computing](https://www.quantamagazine.org/john-preskill-explains-quantum-supremacy-20191002/) over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

<div style="font-size: 0.92em;">

### How I will complete this problem

#### Goal
Build `determine_constant_balanced(f)` so it returns "constant" or "balanced" for a promised four-input Boolean function and provide a clear demonstration and tests.

#### Function Documentation (API)
- **Name:** `determine_constant_balanced`
- **Parameters:** none (evaluates provided callable `f` with 4 boolean arguments)
- **Returns:** string `"constant"` or `"balanced"`
- **Assumption:** `f` satisfies the Deutsch–Jozsa promise (constant or balanced)
- **Libraries used:** Python standard library only (`itertools`)

#### Step-by-step plan
1. List the 16 possible 4-bit inputs deterministically.
2. Evaluate `f` on the first input and record the result.
3. Query further distinct inputs up to 8 more times looking for a different output.
4. If a differing output is found, return `"balanced"` immediately.
5. If 9 calls produced the same output, return `"constant"` (guaranteed under the promise).

#### Why this is correct
- A single differing output proves the function is not constant, therefore balanced.
- Since a balanced function can be True on at most 8 of the 16 inputs (and False on the other 8), seeing the same output on 9 distinct inputs guarantees the function is constant.

#### Complexity notes
- Worst-case deterministic queries: 9 evaluations of `f`.
- Best-case: 2 evaluations (early mismatch).
- Each call to `f` is O(1) for the fixed 4-bit domain, so overall cost is O(1).

#### Example use
```python
from itertools import product
f = random_constant_balanced()
print(determine_constant_balanced(f))
```

#### Testing checklist
- Constant functions (always-True, always-False) are classified as `"constant"`.
- Balanced functions (exactly 8 True outputs) are classified as `\"balanced\"`.
- The implementation never calls `f` more than 9 times.

</div>

In [ ]:
from typing import Callable

def determine_constant_balanced(f: Callable[[bool, bool, bool, bool], bool]) -> str:
    """Classify a promised 4-input Boolean function as 'constant' or 'balanced'.

    Args:
        f (callable): (a: bool, b: bool, c: bool, d: bool) -> bool

    Returns:
        str: 'constant' or 'balanced'

    Notes:
        Deterministic worst-case: 9 calls to `f` to guarantee correctness under the promise.
    """
    import itertools
    all_inputs = list(itertools.product([False, True], repeat=4))
    first_out = f(*all_inputs[0])
    calls = 1
    for inp in all_inputs[1:]:
        out = f(*inp)
        calls += 1
        if out != first_out:
            return "balanced"
        if calls >= 9:
            return "constant"
    return "constant"

In [ ]:
import itertools

# Example: classify a sampled function from Problem 1
f = random_constant_balanced()
print("Sample function classification:", determine_constant_balanced(f))

# Deterministic tests verifying behavior
assert determine_constant_balanced(lambda a,b,c,d: False) == "constant"
assert determine_constant_balanced(lambda a,b,c,d: True) == "constant"
all_inputs = list(itertools.product([False, True], repeat=4))
true_inputs = set(all_inputs[:8])
balanced_f = lambda a,b,c,d: (a,b,c,d) in true_inputs
assert determine_constant_balanced(balanced_f) == "balanced"

# Testing checklist (informal):
# - Constant functions classified as 'constant'
# - Balanced functions classified as 'balanced'
# - Implementation uses at most 9 calls
print("Deterministic tests passed.")


**Efficiency note:** Worst-case deterministic queries: 9 evaluations of `f`. Best-case: 2 evaluations (early mismatch). Each call to `f` is O(1) for the fixed 4-bit domain, so overall cost is O(1) in time and O(1) space.

**Complexity analysis:** The algorithm stops when it finds a differing output (proving 'balanced') or after 9 consistent outputs (proving 'constant'), hence a deterministic upper bound of 9 oracle calls.

#### Problem 3: Quantum Oracles

##### [Deutsch's algorithm](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm) is the simplest example of a [quantum algorithm](https://www.ibm.com/quantum/blog/group-theory) using [superposition](https://scienceexchange.caltech.edu/topics/quantum-science-explained/quantum-superposition) to determine a [global property](https://plato.stanford.edu/archives/fall2008/entries/qt-entangle/#5) of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate [quantum oracles](https://quantumcomputing.stackexchange.com/questions/4625/what-exactly-is-an-oracle/4626#4626) for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

#### Problem 4: Deutsch's Algorithm with Qiskit

##### Use [Qiskit](https://www.ibm.com/quantum/qiskit) to design a [quantum circuit](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information/quantum-circuits/introduction) that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

#### Problem 5: Scaling to the Deutsch–Jozsa Algorithm

##### The [Deutsch–Jozsa algorithm](https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa) generalizes Deutsch's approach to functions with several input bits. Use [Qiskit](https://www.ibm.com/quantum/qiskit) to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.